# FDSN quickstart — recent earthquakes

Fetch recent global **M5+** earthquakes from the USGS ComCat catalog through the `earthlens` FDSN backend, and plot them. FDSN is the first `vector` backend: `download()` returns a pyramids `FeatureCollection` (a `geopandas.GeoDataFrame`), not a raster.

Install with `pip install earthlens[fdsn]` (pulls in `obspy`).

## Setup

Import `EarthLens` and create the output directory. The FDSN backend writes the query result to a GeoPackage on disk and also returns it in memory.

In [ ]:
from pathlib import Path

from earthlens import EarthLens

OUT_DIR = Path('fdsn_output')
OUT_DIR.mkdir(exist_ok=True)

## Build the query

`variables=['USGS']` selects the network; query filters (`min_magnitude`, …) are explicit keyword arguments. Building the request is its own step, kept separate from the network call below.

In [ ]:
fdsn = EarthLens(
    variables=['USGS'],
    data_source='fdsn',
    start='2024-01-01',
    end='2024-01-31',
    aoi=[-180, -90, 180, 90],
    min_magnitude=5.0,
    path=str(OUT_DIR),
)

## Download the events

`download()` runs the live USGS query, writes `fdsn_output/usgs.gpkg`, and returns the events in memory. The call is wrapped so the notebook stays safe under nbval-lax when run offline (the cell never raises).

In [ ]:
events = None
try:
    events = fdsn.download()
    print(f'{len(events)} events')
except Exception as exc:
    print(f'live FDSN query skipped (offline?): {type(exc).__name__}: {exc}')

## Inspect the events

In [ ]:
if events is not None and len(events):
    display(events[['time', 'magnitude', 'depth_km', 'provider', 'geometry']].head())
    print('CRS:', events.crs)

## Plot the epicentres

Marker area scales with magnitude.

In [ ]:
if events is not None and len(events):
    ax = events.plot(markersize=events['magnitude'] ** 2, alpha=0.5, color='firebrick')
    ax.set_title('M5.0+ earthquakes, January 2024 (USGS)')
    ax.set_xlabel('longitude')
    ax.set_ylabel('latitude')